# VoxCPM2 单旁白小说配音（免费 Google Colab）

这是一个**仓库优先**的长文本 TTS 流程：Notebook 会克隆本公开 GitHub 仓库。默认把小说、参考音频、任务清单和成品放在 Google Drive，以便运行时中断后从 `manifest.json` 恢复；Drive 挂载异常时可切换为临时的 Colab 本地存储，仅用于试听。

后续代码修改只需提交 GitHub；在 Colab 重新运行“获取/同步项目代码”即可。不要上传 `voxcpm_novel.py`，也不要把参考音频、书稿或生成结果提交到公开仓库。

开始前请在 Colab 菜单选择“运行时 → 更改运行时类型 → T4 GPU”（免费账户不保证每次分配到 GPU）。只使用你拥有或已获明确授权的声音与文本。

## 流程与关键边界

1. 克隆/同步公开仓库并安装仓库声明的依赖。
2. 选择存储模式：正式长书使用 Google Drive；Drive 挂载异常时使用临时的 Colab 本地存储试听。参考音频和个人书稿始终不进入 GitHub。
3. 在存储模式与“输入与任务配置”两个单元填入任务参数；默认使用仓库内的原创长文本测试稿。
4. 预览章节与分段、创建可续跑任务清单、加载模型。
5. 先生成一个片段试听，再决定是否生成全书和导出 M4B。

> 本 Notebook **不会**调用 `files.upload()`，因此不会为代码或输入自动弹出系统上传窗口。T4 的 FP16 配置只是兼容层，先试听是必要验收。

## 0. 获取/同步项目代码

以后本仓库有代码更新，只需重新运行下一格。该操作只同步 `/content` 的临时代码，不会触碰你的 Drive 输入或已生成成品。

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/MAE5blog/voxcpm2-novel-tts-colab.git"
REPO_REF = "main"
REPO_DIR = Path("/content/voxcpm2-novel-tts-colab")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

revision = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print("已加载仓库代码：", REPO_DIR, "| revision:", revision)


## 1. Colab GPU 健康检查

没有 CUDA 时，请在 Colab 选择 GPU 运行时后重新连接并从本格开始。

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU。请在 Colab 中选择 T4 GPU 后重连运行时。")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
print(f"GPU: {name} | compute capability: {major}.{minor} | VRAM: {vram_gib:.1f} GiB")


## 2. 安装仓库依赖

不升级 Colab 自带的 PyTorch。`requirements-colab.txt` 由仓库版本控制，代码更新时依赖也能随之同步。

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", str(REPO_DIR / "requirements-colab.txt")],
    check=True,
)

import voxcpm
print("VoxCPM package is ready:", getattr(voxcpm, "__version__", "2.0.3"))


## 3. 仓库逻辑测试（无需 GPU 推理）

这一步会运行分章、中文切分、断点清单和参考缓存调用形状测试。它不下载模型、不读取参考音频。

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests"), "-v"], check=True)


## 4. 选择任务存储位置

默认 `drive` 是正式的断点续跑模式：私有输入和成品保存在 `MyDrive/VoxCPM2_Novel/`。若 Drive 当前无法挂载，可改成 `content` 进行一次临时试听；该模式会在运行时回收后丢失输入、清单和成品。本 Notebook 不会调用 `files.upload()` 或自动打开上传窗口。

In [ ]:
# ===== 只在 Drive 挂载失败、且只做试听时改成 content =====
STORAGE_MODE = "drive"  # "drive"（持久断点续跑）或 "content"（临时试听）
# ================================================================

if STORAGE_MODE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        raise RuntimeError(
            "Google Drive 挂载失败。若只是本次试听，将 STORAGE_MODE 改为 'content' 后重跑此格；"
            "正式长书请修复 Drive 授权后保持 drive。"
        ) from exc
    STORAGE_ROOT = Path("/content/drive/MyDrive/VoxCPM2_Novel")
elif STORAGE_MODE == "content":
    STORAGE_ROOT = Path("/content/VoxCPM2_Novel")
else:
    raise ValueError("STORAGE_MODE 只能是 drive 或 content")

INPUTS_DIR = STORAGE_ROOT / "inputs"
JOBS_DIR = STORAGE_ROOT / "jobs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
JOBS_DIR.mkdir(parents=True, exist_ok=True)
print(f"存储模式：{STORAGE_MODE} | 工作目录：{STORAGE_ROOT}")
print("请将参考音频放到：", INPUTS_DIR)
if STORAGE_MODE == "content":
    print("临时输入请在 Colab 左侧 Files 中手动放入上述目录；Notebook 不会弹出上传窗口。")
    print("注意：content 仅用于本次临时试听；运行时重启后内容会丢失。")


## 5. 输入与任务配置（只改这一格）

默认 `repo_demo` 使用仓库内的原创长文本；参考音频从上一格选定的存储位置读取。要使用自己的书稿时，先将文件放到 `inputs/`，把 `BOOK_SOURCE` 改为 `storage` 并填写文件名。改变小说、参考音频、转录、风格或推理参数时，必须使用新的 `JOB_NAME`。

In [ ]:
from voxcpm_novel import RenderOptions

# ==================== 只修改这里 ====================
BOOK_SOURCE = "repo_demo"  # "repo_demo" 或 "storage"
BOOK_FILE = "generated_long_demo.txt"
REFERENCE_AUDIO_FILE = "古龙评书（干声）.flac"

BOOK_TITLE = "雾港邮局的第七码头（VoxCPM2 长文本测试）"
AUTHOR = "仓库内置原创测试文本"
JOB_NAME = "gulong_long_demo_v1"
REFERENCE_TEXT = ""  # 只有逐字准确时才填写
VOICE_STYLE = ""      # 例如："平静、自然、叙述感"
# =====================================================

if BOOK_SOURCE == "repo_demo":
    BOOK_PATH = REPO_DIR / "examples" / BOOK_FILE
elif BOOK_SOURCE == "storage":
    BOOK_PATH = INPUTS_DIR / BOOK_FILE
else:
    raise ValueError("BOOK_SOURCE 只能是 repo_demo 或 storage")

REFERENCE_AUDIO_PATH = INPUTS_DIR / REFERENCE_AUDIO_FILE
if not BOOK_PATH.is_file():
    raise FileNotFoundError(f"小说文件不存在：{BOOK_PATH}")
if not REFERENCE_AUDIO_PATH.is_file():
    raise FileNotFoundError(
        f"未找到参考音频：{REFERENCE_AUDIO_PATH}\n"
        "请将文件放到上一个单元显示的 inputs 目录；不要提交到公开仓库。"
    )

OPTIONS = RenderOptions(
    model_id="openbmb/VoxCPM2",
    model_revision="main",
    target_chars=90,
    hard_chars=160,
    cfg_value=1.6,
    inference_timesteps=10,
    max_len=4096,
    normalize=False,
    optimize=False,
    base_seed=42,
)
print("小说：", BOOK_PATH)
print("参考音频：", REFERENCE_AUDIO_PATH)
print("任务：", JOB_NAME)
print(OPTIONS)


## 6. 导入、预览并创建断点续跑清单

TXT、Markdown、EPUB 和有文字层 PDF 都可导入。先检查章节与切分是否符合预期；扫描 PDF 需要先在外部完成 OCR。

In [ ]:
from voxcpm_novel import VoiceConfig, build_manifest, chunk_text, load_chapters, manifest_summary

chapters = load_chapters(BOOK_PATH)
print(f"检测到 {len(chapters)} 章：")
for chapter in chapters[:8]:
    preview = chapter.text.replace("\n", " ")[:72]
    print(f"  {chapter.index:>3}. {chapter.title}  |  {len(chapter.text)} 字  |  {preview}…")

voice = VoiceConfig(reference_audio=str(REFERENCE_AUDIO_PATH), reference_text=REFERENCE_TEXT, style=VOICE_STYLE)
JOB_DIR = JOBS_DIR / JOB_NAME
manifest = build_manifest(JOB_DIR, chapters, OPTIONS, voice, title=BOOK_TITLE, author=AUTHOR)
print("\n任务摘要：", manifest_summary(manifest))
first_chunks = chunk_text(chapters[0].text, OPTIONS)
print(f"第一章会切成 {len(first_chunks)} 段；前 3 段字数：", [len(item.text) for item in first_chunks[:3]])
print("首段文本：", first_chunks[0].text)


## 7. 下载并加载 VoxCPM2

权重约 5 GB，下载到 `/content` 以避免 Drive I/O。T4（compute capability 7.5）不能原生运行模型默认 BF16，因此只会修改这份临时副本为 FP16；每次新运行时都需要重新下载或复用本次 `/content` 缓存。

In [ ]:
from voxcpm_novel import cuda_preflight, load_voxcpm_model, prepare_colab_model_dir

preflight = cuda_preflight()
print(preflight)
if not preflight.get("cuda_available"):
    raise RuntimeError(preflight.get("reason", "未检测到 CUDA GPU"))

MODEL_DIR = Path("/content/models/VoxCPM2")
model_dir = prepare_colab_model_dir(
    MODEL_DIR,
    model_id=OPTIONS.model_id,
    revision=OPTIONS.model_revision,
    force_float16=preflight.get("needs_fp16_model_copy"),
)
model = load_voxcpm_model(str(model_dir), optimize=OPTIONS.optimize, device="cuda", cache_dir="/content/hf-cache")
print("模型已加载：", model_dir)


## 8. 先试听一个新片段

只处理一个尚未完成的片段，并立即写入正式任务目录。若声音、中文断句或速度不满意，改配置、换一个 `JOB_NAME` 后重新从第 6 步开始。

In [ ]:
from IPython.display import Audio, display
from voxcpm_novel import render_pending_segments

already_done = {item["id"] for item in manifest["segments"] if item["status"] == "completed"}

def preview_progress(record, complete, total):
    print(f"完成 {complete}/{total}：{record['id']}（{len(record['text'])} 字）")

manifest = render_pending_segments(JOB_DIR, manifest, model, progress=preview_progress, max_segments=1)
new_records = [item for item in manifest["segments"] if item["status"] == "completed" and item["id"] not in already_done]
if not new_records:
    raise RuntimeError("没有可试听的待生成片段；任务可能已经全部完成。")
preview_record = new_records[0]
preview_path = JOB_DIR / preview_record["relative_path"]
print("试听片段：", preview_record["id"], preview_record["text"])
display(Audio(filename=str(preview_path)))


## 9. 生成全书（可安全重复运行）

每完成一段都会原子更新 `manifest.json`。正常中断、网络断连或重新连接后，重新同步代码、挂载 Drive、保持同一配置并加载模型，再运行这一格即可跳过已完成段。此功能仅允许 `STORAGE_MODE=drive`；`content` 只用于首段试听。CUDA 真正卡死时应重启运行时，不要强杀 Python。

In [ ]:
from time import monotonic

if STORAGE_MODE != "drive":
    raise RuntimeError("content 模式只允许首段试听；请恢复 STORAGE_MODE='drive' 后再生成全书。")

started = monotonic()

def full_progress(record, complete, total):
    elapsed_min = (monotonic() - started) / 60
    print(f"完成 {complete}/{total} | {elapsed_min:.1f} 分钟 | {record['id']} | {len(record['text'])} 字")

manifest = render_pending_segments(JOB_DIR, manifest, model, progress=full_progress)
print("最终任务摘要：", manifest_summary(manifest))


## 10. 合并章节并导出 M4B

所有段完成后，本格生成 `chapters/chapter_XXX.wav`、`chapters/chapter_XXX.mp3`，以及 `exports/audiobook.m4b`。成品都在 Drive 中；不要把大文件提交到 GitHub。`content` 模式不允许导出，因为它只用于临时试听。

In [ ]:
from voxcpm_novel import export_m4b, merge_completed_chapters

if STORAGE_MODE != "drive":
    raise RuntimeError("content 模式只允许首段试听；请恢复 STORAGE_MODE='drive' 后再导出。")

unfinished = [item["id"] for item in manifest["segments"] if item["status"] != "completed"]
if unfinished:
    raise RuntimeError(f"还有 {len(unfinished)} 段未完成，不能合并。先重新运行第 9 步。")

manifest = merge_completed_chapters(JOB_DIR, manifest)
m4b_path = export_m4b(JOB_DIR, manifest)
print("M4B：", m4b_path)
print("章节 MP3：", JOB_DIR / "chapters")


## 故障恢复与常见问题

- **没有 GPU / 显存不足**：免费 Colab 不保证 T4。遇到 CUDA OOM 会按标点自动二分当前段，最多 3 层；仍失败时新建任务，将 `target_chars` 调到 70、`hard_chars` 调到 120。
- **运行时重启或断连**：`STORAGE_MODE=drive` 时，重新运行第 0、2、4、5、6、7 步（保持同一 `JOB_NAME` 和配置），再运行第 9 步，已完成的 WAV 与清单会复用；`content` 模式不支持断点续跑。
- **找不到参考音频**：将文件放进上一个单元显示的 `inputs/` 目录，并让 `REFERENCE_AUDIO_FILE` 与文件名完全一致。正式运行通常是 `MyDrive/VoxCPM2_Novel/inputs/`；该 Notebook 故意不提供上传按钮。
- **扫描 PDF 报错**：先在外部完成 OCR，生成带文字层 PDF 或 TXT；本 Notebook 不在免费 GPU 上做 OCR。
- **模型下载失败**：重试第 7 步；必要时在 Hugging Face 网页确认 `openbmb/VoxCPM2` 可访问。模型不放在 Drive，以免同步缓慢或损坏。
- **声音不稳定**：优先换成 5–30 秒干净、单人、无音乐的参考音频；有准确转录时再填写 `REFERENCE_TEXT`。不要把长段、过多风格词和高 CFG 一起堆上去。
- **调整风格或音频**：改 `VOICE_STYLE`、参考文本、参考音频或推理参数时，必须换 `JOB_NAME`，避免把不同音色混入一本书。

## 许可证与来源

本仓库采用 AGPL-3.0-only。`voxcpm_novel.py` 的长文本基础设施参考并改写自 [OmniVoice Studio](https://github.com/debpalash/OmniVoice-Studio) 的 AGPL-3.0-only 实现（参考提交 `9f6fb247ab01cef69d79d698649225b69d568ada`）；详见仓库的 `LICENSE` 与 `LICENSES.md`。VoxCPM2 模型和 `voxcpm` 包仍受其各自上游条款约束。